# SolarNL — Opdracht 2: invloed van weersomstandigheden op zonnepanelen

**HvA Energielab / AMOLF · Technische Natuurkunde · 2026–2027**

Dit notebook onderzoekt hoe weersomstandigheden samenhangen met de paneeltemperatuur en de elektrische prestaties van acht zonnepanelen. Het is een vervolg op `SolarNL_EDA.ipynb`.

**Onderzoeksvraag:** Welke invloed hebben verschillende weersomstandigheden op de elektrische prestaties van de acht zonnepanelen, hoe beïnvloeden deze omstandigheden de paneeltemperatuur en welk effect heeft deze temperatuur op de elektrische prestaties?

**Belangrijke beperkingen:** de huidige export beslaat slechts enkele dagen; hiermee kan geen meerjarige degradatie worden aangetoond. Identieke elektrische waarden tussen sommige panelen, ontbrekende meetmomenten en de nog niet bevestigde identificatie en eenheid van `CMP-10` vereisen voorzichtigheid.

**Privacy:** de brondata blijft lokaal. Publiceer geen ruwe data of individuele meetrecords zonder toestemming van de opdrachtgevers.

## 1. Bibliotheken en bronbestand

Het notebook gebruikt de bestaande CSV in dezelfde map. Er worden geen gegevens verwijderd omdat andere variabelen ontbreken. De InfluxDB-export kan meerdere tabellen met herhaalde kopregels bevatten.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from io import StringIO
from IPython.display import display

BESTAND = Path("Datasheet_query_(10).csv")
if not BESTAND.exists():
    raise FileNotFoundError(f"CSV niet gevonden: {BESTAND.resolve()}")

with BESTAND.open("r", encoding="utf-8-sig") as f:
    regels = f.readlines()

tabellen, huidige = [], []
for regel in regels:
    if regel.startswith("#datatype"):
        if huidige:
            tabellen.append(huidige)
        huidige = []
    elif regel.startswith("#") or not regel.strip():
        continue
    else:
        huidige.append(regel)
if huidige:
    tabellen.append(huidige)

frames = []
for tabel in tabellen:
    frame = pd.read_csv(StringIO("".join(tabel)), sep=",")
    frame = frame.loc[:, ~frame.columns.str.startswith("Unnamed")]
    frame = frame.drop(columns=["result", "table", "_start", "_stop"], errors="ignore")
    frame = frame.dropna(how="all")
    if "_time" in frame.columns:
        frame["_time"] = pd.to_datetime(frame["_time"], utc=True, errors="coerce")
    frames.append(frame)

solar_frames = [f for f in frames if "impp" in f.columns]
weather_frames = [f for f in frames if "Temperature-PVMET500" in f.columns]
if not solar_frames or not weather_frames:
    raise ValueError("Solar- of weerdata niet gevonden. Controleer de CSV-structuur.")

df_solar_raw = pd.concat(solar_frames, ignore_index=True)
df_weather = pd.concat(weather_frames, ignore_index=True)
print("Zonnepaneelrijen:", len(df_solar_raw))
print("Weerrijen:", len(df_weather))

## 2. Voorbereiden zonder te streng te filteren

De acht Pt100-kolommen worden samengevoegd. **Alleen een ontbrekende waarde van de variabele die je onderzoekt wordt per analyse uitgesloten.** Dat voorkomt onnodig verlies van temperatuurmetingen.

In [ ]:
df_solar = df_solar_raw.copy()
pt_cols = [c for c in df_solar.columns if c.startswith("Pt100") and c != "Pt100"]
if pt_cols:
    pt = df_solar[pt_cols].apply(pd.to_numeric, errors="coerce").bfill(axis=1).iloc[:, 0]
    if "Pt100" in df_solar.columns:
        df_solar["Pt100"] = pd.to_numeric(df_solar["Pt100"], errors="coerce").combine_first(pt)
    else:
        df_solar["Pt100"] = pt
    df_solar = df_solar.drop(columns=pt_cols)

variabelen = ["Pt100", "ff", "impp", "isc", "vmpp", "voc", "CMP-10", "Ref"]
for col in variabelen:
    if col in df_solar.columns:
        df_solar[col] = pd.to_numeric(df_solar[col], errors="coerce")
for col in ["Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]:
    df_weather[col] = pd.to_numeric(df_weather[col], errors="coerce")

df_solar = df_solar.dropna(subset=["_time", "label"]).sort_values(["label", "_time"]).copy()
df_weather = df_weather.dropna(subset=["_time"]).sort_values("_time").copy()

display(df_solar.groupby("label")[["Pt100", "ff", "impp", "isc", "vmpp", "voc"]].count())
print("Beschikbare weersensorwaarden:")
display(df_weather[["Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]].count())

## 3. Synchronisatie met het weerstation

Weerwaarden worden via de **dichtstbijzijnde meting binnen twee minuten** aan zonnepaneelmetingen gekoppeld. We gebruiken UTC om tijdzone- en wintertijdproblemen te vermijden. Een ontbrekende match blijft ontbrekend.

In [ ]:
weer = (df_weather[["_time", "Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]]
        .drop_duplicates(subset=["_time"])
        .sort_values("_time"))
solar_sorted = df_solar.sort_values("_time").copy()
df = pd.merge_asof(
    solar_sorted, weer, on="_time",
    direction="nearest", tolerance=pd.Timedelta("2min"),
    suffixes=("", "_weer")
)
df["delta_T"] = df["Pt100"] - df["Temperature-PVMET500"]
df["P_mpp"] = df["impp"] * df["vmpp"]
print(f"Gekoppelde rijen: {len(df):,}")
print(f"Met geldige omgevingstemperatuur: {df['Temperature-PVMET500'].notna().sum():,}")
display(df.groupby("label")[["Pt100", "delta_T", "P_mpp"]].count())

## 4. Weersomstandigheden en paneeltemperatuur

De twee interactieve figuren onderzoeken hoe warm de zonnepanelen zijn ten opzichte van de lucht op de meetlocatie.

**Figuur 4a — Temperatuur door de tijd**
- **X-as:** datum en tijd in **UTC**. De tijdstempels geven aan wanneer de sensor een meting registreerde.
- **Y-as:** temperatuur in **°C**. Een waarde van 20 betekent 20 graden Celsius, niet een temperatuurverschil.
- **Gekleurde reeksen:** de **Pt100-paneeltemperatuur** van PV1–PV8; de zwarte lijn is de **PVMET500-omgevingstemperatuur**.
- Zoom door een gebied te slepen, dubbelklik om de schaal te herstellen en klik op de legenda om een paneel te verbergen.

**Figuur 4b — Temperatuurverschil per paneel**
- **X-as:** het paneelnummer PV1–PV8; dit is een **categorie**, geen continue numerieke grootheid.
- **Y-as:** **ΔT = T_paneel − T_omgeving**, in **°C**. Een waarde van +5 betekent dat het paneel 5 °C warmer is dan de lucht; −5 betekent 5 °C kouder.
- De box toont het middelste kwartielbereik (25e–75e percentiel), de middenlijn de mediaan en de punten individuele metingen.

**Interpretatie:** verschillen kunnen samenhangen met instraling, wind, paneelconstructie en sensormontage. De eerder waargenomen sterk negatieve PV8-waarden vereisen controle; trek daar nog geen fysische conclusie uit.

In [ ]:
# Figuur 4a: interactieve tijdreeksen
fig = go.Figure()
for paneel, g in sorted(df.groupby("label")):
    g = g.dropna(subset=["_time", "Pt100"]).sort_values("_time")
    fig.add_trace(go.Scatter(
        x=g["_time"], y=g["Pt100"], mode="lines+markers",
        name=paneel.replace("PvDevice-", "PV"),
        marker=dict(size=4), line=dict(width=1.5),
        hovertemplate="Tijd: %{x|%d-%m-%Y %H:%M} UTC<br>Paneeltemperatuur: %{y:.2f} °C<extra>%{fullData.name}</extra>"
    ))
fig.add_trace(go.Scatter(
    x=weer["_time"], y=weer["Temperature-PVMET500"],
    mode="lines", name="Omgeving (PVMET500)",
    line=dict(color="#1d2430", width=2.5),
    hovertemplate="Tijd: %{x|%d-%m-%Y %H:%M} UTC<br>Omgevingstemperatuur: %{y:.2f} °C<extra></extra>"
))
fig.update_layout(title="4a. Paneeltemperatuur en omgevingstemperatuur",
    xaxis_title="Datum en tijd (UTC)", yaxis_title="Temperatuur (°C)",
    template="plotly_white", hovermode="x unified", height=530,
    legend_title_text="Sensor / paneel")
fig.show()

# Figuur 4b: temperatuurverschil als boxplot
fig = go.Figure()
for paneel, g in sorted(df.groupby("label")):
    g = g.dropna(subset=["delta_T"])
    fig.add_trace(go.Box(
        y=g["delta_T"], name=paneel.replace("PvDevice-", "PV"),
        boxpoints="outliers",
        hovertemplate="ΔT: %{y:.2f} °C<extra>%{fullData.name}</extra>"
    ))
fig.add_hline(y=0, line_dash="dash", line_color="gray",
              annotation_text="Paneel = omgeving")
fig.update_layout(title="4b. Verschil tussen paneel- en omgevingstemperatuur",
    xaxis_title="Zonnepaneel", yaxis_title="ΔT = T_paneel − T_omgeving (°C)",
    template="plotly_white", height=500, showlegend=False)
fig.show()

display(df.groupby("label")["delta_T"].agg(["count", "median", "min", "max"]).round(2))

## 5. Temperatuur en elektrische prestaties

Het elektrisch vermogen bij het maximum power point berekenen we met **Pₘₚₚ = Iₘₚₚ × Vₘₚₚ**. Dit is een berekening uit twee gemeten grootheden, geen onafhankelijk gemeten vermogenssensor.

We tonen **drie afzonderlijke interactieve spreidingsdiagrammen**, zodat je per grootheid kunt inzoomen.

**Voor alle drie de figuren:**
- **X-as:** de gemeten **Pt100-paneeltemperatuur (°C)**. Een punt bij 25 betekent dat het paneel op dat meetmoment 25 °C was.
- **Y-as 5a — Voc (V):** de **openklemspanning**, gemeten in volt. Dit is de spanning bij open elektrisch circuit, niet de werkspanning onder belasting.
- **Y-as 5b — FF (%):** de **fill factor**. Een waarde van 80 betekent ongeveer 80% als de export de gebruikelijke percentageschaal gebruikt; die schaal moet nog bevestigd worden.
- **Y-as 5c — Pₘₚₚ (W):** het berekende maximale elektrische vermogen in **watt**, gelijk aan stroom in ampère maal spanning in volt.
- **Elke stip:** één tijdstip waarop zowel de temperatuur als de elektrische variabele beschikbaar is. De kleur identificeert het paneel.

**Let op:** de figuren tonen samenhang, niet causaliteit. Zoninstraling beïnvloedt zowel de temperatuur als de elektrische prestaties. De eerder vastgestelde overeenkomsten tussen elektrische paneelreeksen maken onderlinge paneelvergelijking voorlopig onzeker. De correlaties onder de figuren zijn beschrijvend en gevoelig voor uitschieters en het beperkte tijdvenster.

In [ ]:
# Drie afzonderlijke interactieve scatterplots
grafieken = [
    ("voc", "5a. Paneeltemperatuur versus openklemspanning", "Openklemspanning Voc (V)", "V"),
    ("ff", "5b. Paneeltemperatuur versus fill factor", "Fill factor FF (vermoedelijk %)", "%"),
    ("P_mpp", "5c. Paneeltemperatuur versus berekend vermogen", "Berekend maximaal vermogen Pₘₚₚ (W)", "W"),
]
for ykolom, titel, yas, eenheid in grafieken:
    fig = go.Figure()
    for paneel, g in sorted(df.groupby("label")):
        g = g.dropna(subset=["Pt100", ykolom])
        fig.add_trace(go.Scatter(
            x=g["Pt100"], y=g[ykolom], mode="markers",
            name=paneel.replace("PvDevice-", "PV"),
            marker=dict(size=7, opacity=0.65),
            customdata=g["_time"].dt.strftime("%d-%m-%Y %H:%M UTC"),
            hovertemplate=(
                "Tijd: %{customdata}<br>"
                "Paneeltemperatuur: %{x:.2f} °C<br>"
                f"{yas}: " + "%{y:.2f} " + eenheid +
                "<extra>%{fullData.name}</extra>"
            )
        ))
    fig.update_layout(title=titel,
        xaxis_title="Paneeltemperatuur Pt100 (°C)", yaxis_title=yas,
        template="plotly_white", height=520,
        legend_title_text="Zonnepaneel")
    fig.show()

for ykolom, _, _, _ in grafieken:
    print(f"\nPearson-correlatie Pt100 versus {ykolom} (beschrijvend):")
    resultaten = {}
    for paneel, g in df.groupby("label"):
        paar = g[["Pt100", ykolom]].dropna()
        resultaten[paneel] = paar["Pt100"].corr(paar[ykolom]) if len(paar) >= 3 else np.nan
    display(pd.Series(resultaten, name="Pearson r").round(3).to_frame())

## 6. CMP-10: vermoedelijke zoninstraling

**CMP-10 is vermoedelijk een Kipp & Zonen CMP10-pyranometer**, die globale kortgolvige zoninstraling meet. De gebruikelijke eenheid is **W/m²**, maar sensoridentificatie, exporteenheid en kalibratie zijn nog niet bevestigd.

**Figuur 6 — CMP-10 door de tijd**
- **X-as:** datum en tijd (**UTC**).
- **Y-as:** geregistreerde **CMP-10-waarde**; vermoedelijk zoninstraling in **W/m²**. Bijvoorbeeld: 500 W/m² betekent 500 joule stralingsenergie per seconde per vierkante meter sensoroppervlak, **als** de eenheid inderdaad W/m² is.
- **De lijn:** de mediaan van gelijktijdige CMP-10-waarden uit de paneelregistraties, om herhaalde waarden niet acht keer te tekenen.

De CMP-10-reeks is een kandidaat voor onderzoek naar de invloed van instraling op temperatuur en vermogen. **Controleer eerst de meeteenheid en eventuele offset**, vooral wanneer 's nachts hoge positieve waarden voorkomen. Zonder die controle berekenen we nog geen instralingsrendement.

In [ ]:
if "CMP-10" in df.columns:
    cmp = (df[["_time", "CMP-10"]].dropna()
           .groupby("_time", as_index=False)["CMP-10"].median()
           .sort_values("_time"))
    fig = go.Figure(go.Scatter(
        x=cmp["_time"], y=cmp["CMP-10"], mode="lines+markers",
        name="CMP-10", line=dict(color="#b88122", width=2),
        marker=dict(size=4),
        hovertemplate="Tijd: %{x|%d-%m-%Y %H:%M} UTC<br>CMP-10: %{y:.2f} (eenheid te verifiëren)<extra></extra>"
    ))
    fig.update_layout(
        title="6. CMP-10 — vermoedelijke zoninstraling (nog te verifiëren)",
        xaxis_title="Datum en tijd (UTC)",
        yaxis_title="CMP-10 (vermoedelijk W/m²; niet bevestigd)",
        template="plotly_white", height=520)
    fig.show()
    display(cmp["CMP-10"].describe().round(2))
else:
    print("Geen CMP-10-kolom aanwezig.")

## 7. Vergelijkbaarheid en datakwaliteit

We controleren (a) het aantal geldige waarden per paneel en (b) of stroommetingen van verschillende panelen op dezelfde tijdstippen identiek zijn. Grote overeenkomsten kunnen duiden op gedeelde registratie; dat moet bij de opdrachtgever worden nagevraagd.

In [ ]:
display(df.groupby("label")[["Pt100", "impp", "isc", "vmpp", "voc", "P_mpp"]].count())
stroom = df.pivot_table(index="_time", columns="label", values="impp", aggfunc="first")
records = []
from itertools import combinations
for a, b in combinations(stroom.columns, 2):
    paar = stroom[[a, b]].dropna()
    if len(paar):
        gelijk = np.isclose(paar[a], paar[b], rtol=0, atol=1e-9)
        records.append({"Paneel A": a, "Paneel B": b,
                        "Gelijktijdig": len(paar), "Identiek": int(gelijk.sum()),
                        "Overeenkomst (%)": round(100*gelijk.mean(), 1)})
vergelijking = pd.DataFrame(records)
if not vergelijking.empty:
    display(vergelijking.sort_values("Overeenkomst (%)", ascending=False).head(15))

## 8. Conclusies en vervolgstappen

Vul na het uitvoeren van de cellen de onderstaande bevindingen aan. Scheid **observaties** van **verklaringen**.

**Voorlopige aandachtspunten**
- PV8 heeft eerder afwijkende negatieve paneeltemperaturen laten zien; controleer de sensorpositie en kalibratie.
- Niet alle elektrische meetwaarden zijn op alle tijdstippen beschikbaar.
- Sommige panelen vertonen sterk overlappende of identieke elektrische reeksen; controleer de meetarchitectuur.
- `CMP-10` is waarschijnlijk zoninstraling (CMP10-pyranometer); verifieer sensoridentificatie, eenheid en kalibratie voordat de waarden fysisch worden geïnterpreteerd.
- Voor degradatieonderzoek zijn gegevens over meerdere jaren en paneelspecificaties nodig.

**Nog in te vullen na uitvoering**
1. Hoe groot zijn de temperatuurverschillen tussen paneel en omgeving?
2. Welke samenhang zien we tussen temperatuur en spanning, fill factor of vermogen?
3. Welke vergelijkingen zijn betrouwbaar gezien ontbrekende en identieke waarden?
4. Welke extra gegevens zijn nodig (instraling, wind, paneeltype, hellingshoek, kalibratie)?

**Publicatie:** neem alleen geaggregeerde en door de opdrachtgever goedgekeurde resultaten over op de GitHub Pages-site.